In [ ]:
"""
Gaussian Blur 실험 결과 시각화 스크립트
all_blur_results.json → 10개 그래프 생성
한글/기호 깨짐 없음 (NanumGothic 사용)
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
import warnings
import os
warnings.filterwarnings('ignore')

# ── 폰트 설정 (Windows / Linux 둘 다 지원) ─────────────────
_CANDIDATE_PATHS = [
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',      # Linux
    r'C:\Windows\Fonts\NanumGothic.ttf',                     # Windows (일반 설치)
    r'C:\Windows\Fonts\NanumGothic-Regular.ttf',
    os.path.expanduser(r'~\AppData\Local\Microsoft\Windows\Fonts\NanumGothic.ttf'),  # Windows (사용자별 설치)
]
_NANUM_PATH = next((p for p in _CANDIDATE_PATHS if os.path.exists(p)), None)

if _NANUM_PATH:
    fm.fontManager.addfont(_NANUM_PATH)
    fm._load_fontmanager(try_read_cache=False)
    FONT = fm.FontProperties(fname=_NANUM_PATH).get_name()
else:
    _installed = {f.name for f in fm.fontManager.ttflist}
    FONT = 'NanumGothic' if 'NanumGothic' in _installed else 'Malgun Gothic'
    print(f'[경고] NanumGothic 파일을 못 찾아 "{FONT}"로 대체합니다. '
          f'폰트가 깨지면 나눔고딕 설치 경로를 확인하세요.')

plt.rcParams.update({
    'font.family': FONT,
    'axes.unicode_minus': False,
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.edgecolor':   'black',
    'axes.labelcolor':  'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'text.color':       'black',
    'grid.color':       '#cccccc',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': 'white',
    'legend.edgecolor': '#cccccc',
    'legend.labelcolor':'black',
})

# ── 경로 설정 (필요하면 여기만 수정) ───────────────────────
JSON_PATH = r".\그림\all_blur_results.json"  # confirmed
OUT       = r".\그림"
os.makedirs(OUT, exist_ok=True)

# ── 데이터 로드 ────────────────────────────────────────────
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

data    = {int(k): v for k, v in raw.items()}
sigmas  = sorted(data.keys())
# 실제 sigma 값 그대로 라벨링 (0이 없으면 자동으로 첫 값이 최소 sigma로 표시됨)
labels  = [f'σ={s}' for s in sigmas]
accs    = [data[s]['acc'] * 100 for s in sigmas]
aucs    = [data[s]['roc_auc'] for s in sigmas]
benign  = [data[s]['per_class_acc']['Bengin cases'] * 100 for s in sigmas]
malig   = [data[s]['per_class_acc']['Malignant cases'] * 100 for s in sigmas]
normal  = [data[s]['per_class_acc']['Normal cases'] * 100 for s in sigmas]

# 색상 팔레트 (정확도에 따라 green→red)
def acc_color(val):
    if val >= 85: return '#2ea043'
    if val >= 75: return '#7ec852'
    if val >= 65: return '#d4e052'
    if val >= 55: return '#e8a838'
    if val >= 45: return '#e86b38'
    return '#c0392b'

BAR_COLORS = [acc_color(a) for a in accs]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 블러 강도별 AI 정확도 (막대 그래프)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(22, 11))
bars = ax.bar(labels, accs, color=BAR_COLORS, width=0.6, zorder=3)

for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=35, fontweight='bold', color='black')

ax.axhline(max(accs), color='#ff6b6b', linestyle='--', lw=1.5, label=f'최고 {max(accs):.1f}%')
ax.axhline(min(accs), color='#4ecdc4', linestyle='--', lw=1.5, label=f'최저 {min(accs):.1f}%')
ax.axhline(80,        color='gray',   linestyle=':',  lw=1.2, label='80% 기준선')

ax.set_xlabel('블러 Sigma (σ)', fontsize=45)
ax.set_ylabel('정확도 (%)',      fontsize=45)
ax.tick_params(labelsize=35)
ax.set_ylim(0, 115)
ax.legend(fontsize=40)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig(f'{OUT}/graph1_accuracy_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph1 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. ROC-AUC 꺾은선
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(18, 8))
ax.plot(labels, aucs, color='#7ca4f4', lw=2.5, marker='o', markersize=10,
        markerfacecolor='white', markeredgecolor='#7ca4f4', zorder=4)

for i, (lbl, val) in enumerate(zip(labels, aucs)):
    ax.annotate(f'{val:.4f}', (lbl, val), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=14, color='black')

ax.axhline(0.9, color='#2ea043', linestyle='--', lw=1.5, label='ROC 0.9 기준')
ax.axhline(0.8, color='#d4a017', linestyle='--', lw=1.5, label='ROC 0.8 기준')

ax.set_title('블러 강도별 ROC-AUC (Blur Sigma vs ROC-AUC)', fontsize=22, pad=16)
ax.set_xlabel('블러 Sigma (σ)', fontsize=17)
ax.set_ylabel('ROC-AUC',        fontsize=17)
ax.tick_params(labelsize=15)
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=14)
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig(f'{OUT}/graph2_roc_auc.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph2 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. 클래스별 정확도 (grouped bar)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(20, 9))
x = np.arange(len(labels))
w = 0.26

ax.bar(x - w, benign, w, label='양성 Benign',    color='#e07070', zorder=3)
ax.bar(x,     malig,  w, label='악성 Malignant',  color='#e0c040', zorder=3)
ax.bar(x + w, normal, w, label='정상 Normal',     color='#60c060', zorder=3)
ax.axhline(80, color='gray', linestyle='--', lw=1.2, label='80% 기준선')

ax.set_title('클래스별 정확도 (Per-Class Accuracy by Blur Sigma)', fontsize=22, pad=16)
ax.set_xlabel('블러 Sigma (σ)', fontsize=17)
ax.set_ylabel('정확도 (%)',      fontsize=17)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=14)
ax.tick_params(axis='y', labelsize=15)
ax.set_ylim(0, 115)
ax.legend(fontsize=14)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig(f'{OUT}/graph3_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph3 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4. 보안성 vs AI 성능 트레이드오프 (scatter)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
blur_strength = [np.log2(max(s, 0.5)) * 5 for s in sigmas]
cmap = LinearSegmentedColormap.from_list('rg', ['#c0392b','#e8a838','#2ea043'])
norm_vals = [(a - min(accs)) / (max(accs) - min(accs)) for a in accs]

fig, ax = plt.subplots(figsize=(13, 10))
for i, (bs, ac, nv, lbl) in enumerate(zip(blur_strength, accs, norm_vals, labels)):
    color = cmap(nv)
    ax.scatter(bs, ac, s=300, color=color, zorder=4, edgecolors='white', linewidths=1)
    ax.annotate(lbl.replace('\n',''), (bs, ac), textcoords='offset points',
                xytext=(8, 4), fontsize=12, color='black')

sm = plt.cm.ScalarMappable(cmap=cmap,
     norm=plt.Normalize(vmin=min(accs), vmax=max(accs)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('AI 정확도 / Accuracy (%)', fontsize=14, color='black')
cbar.ax.yaxis.set_tick_params(color='black', labelcolor='black', labelsize=12)

ax.set_title('보안성 vs AI 성능 트레이드오프 (Security vs Performance)', fontsize=20, pad=16)
ax.set_xlabel('블러 강도 / Blur Strength (높을수록 보안 강함) →', fontsize=15)
ax.set_ylabel('AI 정확도 (%)', fontsize=15)
ax.tick_params(labelsize=14)
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig(f'{OUT}/graph4_tradeoff.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph4 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5. 정확도 변화 추세 (multi-line)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(20, 9))
x = np.arange(len(labels))
ax.plot(x, accs,   color='black', lw=3,   marker='o', markersize=10, label='전체 정확도', zorder=4)
ax.plot(x, benign, color='#e07070', lw=2,   marker='s', markersize=9,  linestyle='--', label='양성 Benign', zorder=3)
ax.plot(x, malig,  color='#e0c040', lw=2,   marker='^', markersize=9,  linestyle='--', label='악성 Malignant', zorder=3)
ax.plot(x, normal, color='#60c060', lw=2,   marker='D', markersize=9,  linestyle='--', label='정상 Normal', zorder=3)
ax.axhline(80, color='gray', linestyle=':', lw=1.2, label='80% 기준선')

for i, val in enumerate(accs):
    ax.annotate(f'{val:.1f}%', (x[i], val), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=12, color='#ffd700')

ax.set_title('블러 강도에 따른 정확도 변화 추세 (Accuracy Trend by Blur Sigma)', fontsize=22, pad=16)
ax.set_xlabel('블러 Sigma (왼쪽=약한 블러 / 오른쪽=강한 블러)', fontsize=16)
ax.set_ylabel('정확도 (%)', fontsize=16)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=14)
ax.tick_params(axis='y', labelsize=15)
ax.set_ylim(-5, 115)
ax.legend(fontsize=14)
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig(f'{OUT}/graph5_trend.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph5 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6. Sigma vs Accuracy (log scale)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(18, 8))
ax.plot(labels, accs, color='black', lw=3, marker='o', markersize=10,
        markerfacecolor='white', markeredgecolor='black', zorder=4)

for i, (lbl, val) in enumerate(zip(labels, accs)):
    ax.annotate(f'{val:.1f}%', (lbl, val), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=15, color='black')

ax.set_title('블러 Sigma vs AI 정확도 (log scale)',
             fontsize=22, pad=16)
ax.set_xlabel('블러 Sigma (log scale) →', fontsize=17)
ax.set_ylabel('정확도 (%)',               fontsize=17)
ax.tick_params(labelsize=15)
ax.set_ylim(0, 110)
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig(f'{OUT}/graph6_sigma_vs_acc.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph6 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7. 클래스별 정확도 히트맵
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
heatmap_data = np.array([benign, malig, normal])
row_labels   = ['양성\n(Benign)', '악성\n(Malignant)', '정상\n(Normal)']

cmap_rg = LinearSegmentedColormap.from_list('rg', ['#c0392b','#e8a838','#2ea043'])
fig, ax = plt.subplots(figsize=(20, 7))
im = ax.imshow(heatmap_data, cmap=cmap_rg, aspect='auto', vmin=0, vmax=100)

for i in range(3):
    for j in range(len(labels)):
        val = heatmap_data[i, j]
        tc = 'black' if 60 < val < 90 else 'white'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                fontsize=17, fontweight='bold', color=tc)

ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=15)
ax.set_yticks(range(3));           ax.set_yticklabels(row_labels, fontsize=16)
ax.set_title('클래스별 정확도 히트맵 (Per-Class Accuracy Heatmap)', fontsize=22, pad=16)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('정확도 / Accuracy (%)', fontsize=14, color='black')
cbar.ax.yaxis.set_tick_params(color='black', labelcolor='black', labelsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}/graph7_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph7 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 8. 레이더 차트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 실제 존재하는 sigma 값 중에서 4개(처음/중간1/중간2/끝)를 자동 선택
_n = len(sigmas)
_idxs = sorted(set([0, _n // 3, 2 * _n // 3, _n - 1]))
radar_sigmas = [sigmas[i] for i in _idxs]
radar_colors = ['#ffd700', '#2ea043', '#e07070', '#7ca4f4'][:len(radar_sigmas)]
radar_names  = [f'σ={s}' for s in radar_sigmas]

categories     = ['Benign', '전체 정확도', 'ROC-AUC×100', 'Normal', 'Malignant']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(11, 10), subplot_kw={'polar': True})
ax.set_facecolor('white')
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], categories, fontsize=16, color='black')
ax.set_rlabel_position(30)
plt.yticks([20,40,60,80,100], ['20','40','60','80','100'], fontsize=11, color='black')
ax.set_ylim(0, 100)

for sig, color, name in zip(radar_sigmas, radar_colors, radar_names):
    d = data[sig]
    vals = [
        d['per_class_acc']['Bengin cases'] * 100,
        d['acc'] * 100,
        d['roc_auc'] * 100,
        d['per_class_acc']['Normal cases'] * 100,
        d['per_class_acc']['Malignant cases'] * 100,
    ]
    vals += vals[:1]
    ax.plot(angles, vals, color=color, lw=2.5, marker='o', markersize=6, label=name)
    ax.fill(angles, vals, color=color, alpha=0.08)

ax.set_title('주요 블러 강도별 성능 레이더 차트 (Radar Chart by Blur Sigma)',
             fontsize=18, pad=28)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=14)
ax.grid(color='#ccc', linestyle='--', alpha=0.6)
ax.spines['polar'].set_color('#999')
plt.tight_layout()
plt.savefig(f'{OUT}/graph8_radar.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph8 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 9. Confusion Matrix 비교 (최소 sigma vs 최대 sigma)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def draw_cm(ax, cm, title, acc):
    cm_arr = np.array(cm)
    cmap_b = plt.cm.Blues
    im = ax.imshow(cm_arr, cmap=cmap_b, aspect='auto')
    cl = ['Benign', 'Malignant', 'Normal']
    ax.set_xticks(range(3)); ax.set_xticklabels(cl, fontsize=14)
    ax.set_yticks(range(3)); ax.set_yticklabels(cl, fontsize=14)
    ax.set_xlabel('AI 예측 (Predicted)', fontsize=14)
    ax.set_ylabel('실제 정답 (Actual)',   fontsize=14)
    ax.set_title(f'{title}', fontsize=16, pad=10)
    thresh = cm_arr.max() / 2
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm_arr[i, j]), ha='center', va='center',
                    fontsize=18, fontweight='bold',
                    color='black' if cm_arr[i, j] > thresh else 'black')
    return im

_min_sig, _max_sig = sigmas[0], sigmas[-1]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')
fig.suptitle('Confusion Matrix: 약한 블러 vs 강한 블러 비교', fontsize=22, y=1.02)

draw_cm(axes[0], data[_min_sig]['cm'], f'약한 블러 (σ={_min_sig})', data[_min_sig]['acc']*100)
im2 = draw_cm(axes[1], data[_max_sig]['cm'], f'최강 블러 (σ={_max_sig})', data[_max_sig]['acc']*100)

cbar = fig.colorbar(im2, ax=axes, fraction=0.03, pad=0.04)
cbar.ax.yaxis.set_tick_params(color='black', labelcolor='black', labelsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}/graph9_confusion.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph9 done')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 10. 종합 대시보드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig = plt.figure(figsize=(24, 16))
fig.patch.set_facecolor('white')
fig.suptitle('블러 강도별 AI 성능 종합 대시보드 (Gaussian Blur AI Performance Dashboard)',
    fontsize=20, y=0.99
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.93, bottom=0.06, left=0.06, right=0.97)

# [0,0] 정확도 막대
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(labels, accs, color=BAR_COLORS, width=0.65, zorder=3)
for bar, val in zip(bars, accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_title('블러 강도별 정확도', fontsize=15, pad=8)
ax1.set_ylabel('정확도 (%)', fontsize=13)
ax1.tick_params(labelsize=11)
ax1.set_ylim(0, 110)
ax1.grid(axis='y', zorder=0)

# [0,1] 실험 결론 텍스트 박스 (실제 데이터로 자동 계산)
ax_txt = fig.add_subplot(gs[0, 1])
ax_txt.set_facecolor('white')
ax_txt.axis('off')

_best_idx  = int(np.argmax(accs))
_worst_idx = int(np.argmin(accs))
conclusion = (
    f"▶ 실험 조건\n"
    f"   Sigma: {len(sigmas)}개 조건 실험\n\n"
    f"▶ 최고 성능\n"
    f"   {labels[_best_idx]}:  {accs[_best_idx]:.1f}%\n"
    f"   ROC-AUC: {aucs[_best_idx]:.4f}\n\n"
    f"▶ 최저 성능\n"
    f"   {labels[_worst_idx]}  정확도: {accs[_worst_idx]:.1f}%\n\n"
    f"▶ 핵심 관찰\n"
    f"   악성(Malignant) 클래스\n"
    f"   최저: {min(malig):.1f}%"
)
ax_txt.text(0.05, 0.97, conclusion, transform=ax_txt.transAxes,
            fontsize=13, va='top', linespacing=1.6, color='black', family=FONT)
ax_txt.set_title('실험 결론', fontsize=15, pad=8)

# [0,2] ROC-AUC
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(range(len(labels)), aucs, color='#7ca4f4', lw=2.5,
         marker='o', markersize=7, markerfacecolor='white', zorder=4)
for i, v in enumerate(aucs):
    ax2.annotate(f'{v:.3f}', (i, v), textcoords='offset points',
                 xytext=(0,8), ha='center', fontsize=10, color='black')
ax2.axhline(0.9, color='#2ea043', linestyle='--', lw=1.2)
ax2.axhline(0.8, color='#d4a017', linestyle='--', lw=1.2)
ax2.set_title('ROC-AUC', fontsize=15, pad=8)
ax2.set_ylabel('ROC-AUC', fontsize=13)
ax2.set_xticks(range(len(labels))); ax2.set_xticklabels(labels, fontsize=10)
ax2.set_ylim(0.5, 1.05)
ax2.grid(zorder=0)

# [1,0] 클래스별 정확도
ax3 = fig.add_subplot(gs[1, 0])
x = np.arange(len(labels))
ax3.plot(x, benign, color='#e07070', lw=2, marker='s', markersize=7, linestyle='--', label='양성(Benign)')
ax3.plot(x, malig,  color='#e0c040', lw=2, marker='^', markersize=7, linestyle='--', label='악성(Malignant)')
ax3.plot(x, normal, color='#60c060', lw=2, marker='D', markersize=7, linestyle='--', label='정상(Normal)')
ax3.set_title('클래스별 정확도', fontsize=15, pad=8)
ax3.set_ylabel('정확도 (%)', fontsize=13)
ax3.set_xticks(x); ax3.set_xticklabels(labels, fontsize=10)
ax3.set_ylim(0, 110)
ax3.legend(fontsize=11)
ax3.grid(zorder=0)

# [1,1] 히트맵
ax4 = fig.add_subplot(gs[1, 1])
cmap_rg2 = LinearSegmentedColormap.from_list('rg', ['#c0392b','#e8a838','#2ea043'])
hm = np.array([benign, malig, normal])
im4 = ax4.imshow(hm, cmap=cmap_rg2, aspect='auto', vmin=0, vmax=100)
for i in range(3):
    for j in range(len(labels)):
        v = hm[i,j]
        tc = 'black' if 50 < v < 85 else 'white'
        ax4.text(j, i, f'{v:.0f}%', ha='center', va='center', fontsize=9, color=tc, fontweight='bold')
ax4.set_xticks(range(len(labels))); ax4.set_xticklabels(labels, fontsize=9)
ax4.set_yticks([0,1,2]); ax4.set_yticklabels(['양성','악성','정상'], fontsize=13)
ax4.set_title('클래스별 정확도 히트맵', fontsize=15, pad=8)

# [1,2] 트레이드오프 scatter
ax5 = fig.add_subplot(gs[1, 2])
bs_vals = [np.log2(max(s, 0.5)) * 5 for s in sigmas]
norm_v  = [(a - min(accs)) / (max(accs) - min(accs)) for a in accs]
for bsv, ac, nv, lbl in zip(bs_vals, accs, norm_v, labels):
    color = cmap_rg2(nv)
    ax5.scatter(bsv, ac, s=120, color=color, zorder=4, edgecolors='white', lw=0.8)
    ax5.annotate(lbl.replace('\n',''), (bsv, ac), textcoords='offset points',
                 xytext=(5,3), fontsize=10, color='black')
ax5.set_title('보안 vs 성능', fontsize=15, pad=8)
ax5.set_xlabel('블러 강도 →', fontsize=13)
ax5.set_ylabel('AI(정확도%)', fontsize=13)
ax5.grid(zorder=0)

plt.savefig(f'{OUT}/graph10_dashboard.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph10 done')
print('\n✅ 모든 그래프 생성 완료!')


In [ ]:
"""
Block Encryption 실험 결과 시각화 스크립트
all_block_results.json → graph1 (정확도 막대그래프)만 생성
"""

import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
import os
warnings.filterwarnings('ignore')

# ── 폰트 설정 (Windows / Linux 둘 다 지원) ─────────────────
_CANDIDATE_PATHS = [
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    r'C:\Windows\Fonts\NanumGothic.ttf',
    r'C:\Windows\Fonts\NanumGothic-Regular.ttf',
    os.path.expanduser(r'~\AppData\Local\Microsoft\Windows\Fonts\NanumGothic.ttf'),
]
_NANUM_PATH = next((p for p in _CANDIDATE_PATHS if os.path.exists(p)), None)

if _NANUM_PATH:
    fm.fontManager.addfont(_NANUM_PATH)
    fm._load_fontmanager(try_read_cache=False)
    FONT = fm.FontProperties(fname=_NANUM_PATH).get_name()
else:
    _installed = {f.name for f in fm.fontManager.ttflist}
    FONT = 'NanumGothic' if 'NanumGothic' in _installed else 'Malgun Gothic'
    print(f'[경고] NanumGothic 파일을 못 찾아 "{FONT}"로 대체합니다.')

plt.rcParams.update({
    'font.family': FONT,
    'axes.unicode_minus': False,
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.edgecolor':   'black',
    'axes.labelcolor':  'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'text.color':       'black',
    'grid.color':       '#cccccc',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': 'white',
    'legend.edgecolor': '#cccccc',
    'legend.labelcolor':'black',
})

# ── 경로 설정 (필요하면 여기만 수정) ───────────────────────
JSON_PATH = r".\그림\all_block_results.json"
OUT       = r".\그림"
os.makedirs(OUT, exist_ok=True)

# ── 데이터 로드 ────────────────────────────────────────────
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

data = {int(k): v for k, v in raw.items()}
# block size는 큰 값(원본에 가까움)부터 작은 값(강한 암호화) 순서로 정렬
block_sizes = sorted(data.keys(), reverse=True)

def block_label(b):
    if b == max(block_sizes):
        return '원본'
    if b == 1:
        return '픽셀\n1×1'
    return f'{b}×{b}'

labels = [block_label(b) for b in block_sizes]
accs   = [data[b]['acc'] * 100 for b in block_sizes]

# 색상 팔레트 (정확도에 따라 green→red 계열, 이미지와 동일한 룩)
def acc_color(val):
    if val >= 90: return '#e0602a'   # 진한 주황빨강
    if val >= 80: return '#e0602a'
    if val >= 75: return '#e8a838'   # 주황
    if val >= 65: return '#d4d838'   # 연두노랑
    if val >= 58: return '#8ec13a'   # 연두
    return '#2ea05a'                 # 초록

BAR_COLORS = [acc_color(a) for a in accs]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 블록 크기별 AI 정확도 (막대 그래프)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(22, 11))
bars = ax.bar(labels, accs, color=BAR_COLORS, width=0.6, zorder=3)

for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=35, fontweight='bold', color='black')

ax.axhline(max(accs), color='#ff6b6b', linestyle='--', lw=1.5, label=f'최고 {max(accs):.1f}%')
ax.axhline(min(accs), color='#4ecdc4', linestyle='--', lw=1.5, label=f'최저 {min(accs):.1f}%')
ax.axhline(80,        color='gray',   linestyle=':',  lw=1.2, label='80% 기준선')

ax.set_xlabel('블록 크기', fontsize=45)
ax.set_ylabel('정확도 (%)', fontsize=45)
ax.tick_params(labelsize=30)
ax.set_ylim(0, 115)
ax.legend(fontsize=40)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig(f'{OUT}/graph1_accuracy_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('graph1 done')
